# Data exploration notebook

This is the starting notebook of the project: its goal is to understand the structure
of the data before using it for any analysis.

All data comes from [FastF1](https://docs.fastf1.dev/), a Python library that gives access
to Formula 1 timing, tyre and telemetry data. The data is explored following the library's
hierarchy, from the season calendar down to a single event and its sessions, while lap-by-lap
data and telemetry are covered in the race analysis notebook.

## Setup
Imports and FastF1 cache configuration: downloaded data is stored in the `cache/`
folder at the repo root, so each session is only downloaded once.

In [ ]:
import fastf1
import pandas as pd
import datetime

from f1analysis.data import enable_cache

fastf1.set_log_level("WARNING")

enable_cache()
pd.set_option("display.max_columns", None)

## Available seasons

According to the documentation, timing data, session
information, car telemetry and position data are available from 2018 onwards.

`get_event_schedule(year)` is the library's top-level entry point: it only needs a year
and returns that season's calendar. Let's start by loading every season from 2018 to the current one
and counting how many events each contains.

In [36]:
current_year = datetime.date.today().year
years = range(2018, current_year + 1)

schedules = {year: fastf1.get_event_schedule(year) for year in years}

pd.Series({year: len(schedule) for year, schedule in schedules.items()}, name="Events")

2018    21
2019    21
2020    19
2021    23
2022    24
2023    23
2024    25
2025    25
2026    25
Name: Events, dtype: int64

These counts are higher than the actual number of Grands Prix in some seasons
(e.g. 2020 had 17 races, not 19). By default `get_event_schedule` also returns
pre-season testing events (`include_testing=True`), which have `RoundNumber == 0`.
Let's check whether they explain the difference by counting Grands Prix and testing
events separately.

In [37]:
pd.DataFrame({
    "GPs": {year: (s["RoundNumber"] > 0).sum() for year, s in schedules.items()},
    "Testing":     {year: (s["RoundNumber"] == 0).sum() for year, s in schedules.items()},
})

,GPs,Testing
2018,21,0
2019,21,0
2020,17,2
2021,22,1
2022,22,2
2023,22,1
2024,24,1
2025,24,1
2026,23,2


Testing events explain the difference: once they are removed, the counts match the
real number of Grands Prix, from 17 (2020) to 24 (2024–2025). FastF1 lists testing
only from 2020 onwards.

Since testing is not part of the championship, it is excluded from now on.

In [38]:
schedules = {year: s[s["RoundNumber"] > 0] for year, s in schedules.items()}

## Season calendar

With the available seasons defined, the next question is what a single season calendar
contains: the 2025 season is used as an example, since it is the most recent complete one.

In [39]:
season = schedules[2025]
season

,RoundNumber,Country,Location,OfficialEventName,EventDate,EventName,EventFormat,Session1,Session1Date,Session1DateUtc,Session2,Session2Date,Session2DateUtc,Session3,Session3Date,Session3DateUtc,Session4,Session4Date,Session4DateUtc,Session5,Session5Date,Session5DateUtc,F1ApiSupport
1,1,Australia,Melbourne,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,2025-03-16,Australian Grand Prix,conventional,Practice 1,2025-03-14 12:30:00+11:00,2025-03-14 01:30:00,Practice 2,2025-03-14 16:00:00+11:00,2025-03-14 05:00:00,Practice 3,2025-03-15 12:30:00+11:00,2025-03-15 01:30:00,Qualifying,2025-03-15 16:00:00+11:00,2025-03-15 05:00:00,Race,2025-03-16 15:00:00+11:00,2025-03-16 04:00:00,True
2,2,China,Shanghai,FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2025,2025-03-23,Chinese Grand Prix,sprint_qualifying,Practice 1,2025-03-21 11:30:00+08:00,2025-03-21 03:30:00,Sprint Qualifying,2025-03-21 15:30:00+08:00,2025-03-21 07:30:00,Sprint,2025-03-22 11:00:00+08:00,2025-03-22 03:00:00,Qualifying,2025-03-22 15:00:00+08:00,2025-03-22 07:00:00,Race,2025-03-23 15:00:00+08:00,2025-03-23 07:00:00,True
3,3,Japan,Suzuka,FORMULA 1 LENOVO JAPANESE GRAND PRIX 2025,2025-04-06,Japanese Grand Prix,conventional,Practice 1,2025-04-04 11:30:00+09:00,2025-04-04 02:30:00,Practice 2,2025-04-04 15:00:00+09:00,2025-04-04 06:00:00,Practice 3,2025-04-05 11:30:00+09:00,2025-04-05 02:30:00,Qualifying,2025-04-05 15:00:00+09:00,2025-04-05 06:00:00,Race,2025-04-06 14:00:00+09:00,2025-04-06 05:00:00,True
4,4,Bahrain,Sakhir,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2025,2025-04-13,Bahrain Grand Prix,conventional,Practice 1,2025-04-11 14:30:00+03:00,2025-04-11 11:30:00,Practice 2,2025-04-11 18:00:00+03:00,2025-04-11 15:00:00,Practice 3,2025-04-12 15:30:00+03:00,2025-04-12 12:30:00,Qualifying,2025-04-12 19:00:00+03:00,2025-04-12 16:00:00,Race,2025-04-13 18:00:00+03:00,2025-04-13 15:00:00,True
5,5,Saudi Arabia,Jeddah,FORMULA 1 STC SAUDI ARABIAN GRAND PRIX 2025,2025-04-20,Saudi Arabian Grand Prix,conventional,Practice 1,2025-04-18 16:30:00+03:00,2025-04-18 13:30:00,Practice 2,2025-04-18 20:00:00+03:00,2025-04-18 17:00:00,Practice 3,2025-04-19 16:30:00+03:00,2025-04-19 13:30:00,Qualifying,2025-04-19 20:00:00+03:00,2025-04-19 17:00:00,Race,2025-04-20 20:00:00+03:00,2025-04-20 17:00:00,True
6,6,United States,Miami Gardens,FORMULA 1 CRYPTO.COM MIAMI GRAND PRIX 2025,2025-05-04,Miami Grand Prix,sprint_qualifying,Practice 1,2025-05-02 12:30:00-04:00,2025-05-02 16:30:00,Sprint Qualifying,2025-05-02 16:30:00-04:00,2025-05-02 20:30:00,Sprint,2025-05-03 12:00:00-04:00,2025-05-03 16:00:00,Qualifying,2025-05-03 16:00:00-04:00,2025-05-03 20:00:00,Race,2025-05-04 16:00:00-04:00,2025-05-04 20:00:00,True
7,7,Italy,Imola,FORMULA 1 AWS GRAN PREMIO DEL MADE IN ITALY E ...,2025-05-18,Emilia Romagna Grand Prix,conventional,Practice 1,2025-05-16 13:30:00+02:00,2025-05-16 11:30:00,Practice 2,2025-05-16 17:00:00+02:00,2025-05-16 15:00:00,Practice 3,2025-05-17 12:30:00+02:00,2025-05-17 10:30:00,Qualifying,2025-05-17 16:00:00+02:00,2025-05-17 14:00:00,Race,2025-05-18 15:00:00+02:00,2025-05-18 13:00:00,True
8,8,Monaco,Monaco,FORMULA 1 TAG HEUER GRAND PRIX DE MONACO 2025,2025-05-25,Monaco Grand Prix,conventional,Practice 1,2025-05-23 13:30:00+02:00,2025-05-23 11:30:00,Practice 2,2025-05-23 17:00:00+02:00,2025-05-23 15:00:00,Practice 3,2025-05-24 12:30:00+02:00,2025-05-24 10:30:00,Qualifying,2025-05-24 16:00:00+02:00,2025-05-24 14:00:00,Race,2025-05-25 15:00:00+02:00,2025-05-25 13:00:00,True
9,9,Spain,Barcelona,FORMULA 1 ARAMCO GRAN PREMIO DE ESPAÑA 2025,2025-06-01,Spanish Grand Prix,conventional,Practice 1,2025-05-30 13:30:00+02:00,2025-05-30 11:30:00,Practice 2,2025-05-30 17:00:00+02:00,2025-05-30 15:00:00,Practice 3,2025-05-31 12:30:00+02:00,2025-05-31 10:30:00,Qualifying,2025-05-31 16:00:00+02:00,2025-05-31 14:00:00,Race,2025-06-01 15:00:00+02:00,2025-06-01 13:00:00,True
10,10,Canada,Montréal,FORMULA 1 PIRELLI GRAND PRIX DU CANADA 2025,2025-06-15,Canadian Grand Prix,conventional,Practice 1,2025-06-13 13:30:00-04:00,202

Each row represents a Grand Prix. The columns of the calendar can be grouped as follows:

- **Identity**: `RoundNumber`, `EventName`, `OfficialEventName`
- **Location**: `Country`, `Location`
- **Date**: `EventDate`
- **Format**: `EventFormat`
- **Sessions**: `Session1` to `Session5`, each with its local and UTC date
- **Data availability**: `F1ApiSupport`

Each group is explored below, starting from the identity and location of each event.

### Identity and location

In [40]:
season[["RoundNumber", "EventName", "OfficialEventName", "Country", "Location"]]

,RoundNumber,EventName,OfficialEventName,Country,Location
1,1,Australian Grand Prix,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,Australia,Melbourne
2,2,Chinese Grand Prix,FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2025,China,Shanghai
3,3,Japanese Grand Prix,FORMULA 1 LENOVO JAPANESE GRAND PRIX 2025,Japan,Suzuka
4,4,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2025,Bahrain,Sakhir
5,5,Saudi Arabian Grand Prix,FORMULA 1 STC SAUDI ARABIAN GRAND PRIX 2025,Saudi Arabia,Jeddah
6,6,Miami Grand Prix,FORMULA 1 CRYPTO.COM MIAMI GRAND PRIX 2025,United States,Miami Gardens
7,7,Emilia Romagna Grand Prix,FORMULA 1 AWS GRAN PREMIO DEL MADE IN ITALY E ...,Italy,Imola
8,8,Monaco Grand Prix,FORMULA 1 TAG HEUER GRAND PRIX DE MONACO 2025,Monaco,Monaco
9,9,Spanish Grand Prix,FORMULA 1 ARAMCO GRAN PREMIO DE ESPAÑA 2025,Spain,Barcelona
10,10,Canadian Grand Prix,FORMULA 1 PIRELLI GRAND PRIX DU CANADA 2025,Canada,Montréal


- `EventName` is the clearest way to identify a Grand Prix: short and consistent across seasons.
- `OfficialEventName` includes the title sponsor, so it changes over the years and is not useful for analysis.
- `Location` is the name of the circuit's town, often the name used in everyday language
  (e.g. Monza, Imola).
- `RoundNumber` is the position of the event in the season.

Together, year and `RoundNumber` identify an event without ambiguity. Names and locations
might not: the next cell checks whether a location can host more than one event in the same season.

In [41]:
for year, s in schedules.items():
    repeated = s[s["Location"].duplicated(keep=False)]
    if not repeated.empty:
        print(year, repeated[["RoundNumber", "EventName", "Location"]].to_dict("records"))

2020 [{'RoundNumber': 1, 'EventName': 'Austrian Grand Prix', 'Location': 'Spielberg'}, {'RoundNumber': 2, 'EventName': 'Styrian Grand Prix', 'Location': 'Spielberg'}, {'RoundNumber': 4, 'EventName': 'British Grand Prix', 'Location': 'Silverstone'}, {'RoundNumber': 5, 'EventName': '70th Anniversary Grand Prix', 'Location': 'Silverstone'}, {'RoundNumber': 15, 'EventName': 'Bahrain Grand Prix', 'Location': 'Sakhir'}, {'RoundNumber': 16, 'EventName': 'Sakhir Grand Prix', 'Location': 'Sakhir'}]
2021 [{'RoundNumber': 8, 'EventName': 'Styrian Grand Prix', 'Location': 'Spielberg'}, {'RoundNumber': 9, 'EventName': 'Austrian Grand Prix', 'Location': 'Spielberg'}]


In 2020 and 2021, some circuits hosted two Grands Prix in the same season (Spielberg,
Silverstone and Sakhir), always under different event names. Therefore, `Location` is not a reliable identifier, and from now on, an event is identified
by **year and `RoundNumber`**, while `EventName` is used for display (e.g. in the app menus).

### Date and format

In [42]:
season[["RoundNumber", "EventName", "EventDate", "EventFormat"]]

,RoundNumber,EventName,EventDate,EventFormat
1,1,Australian Grand Prix,2025-03-16,conventional
2,2,Chinese Grand Prix,2025-03-23,sprint_qualifying
3,3,Japanese Grand Prix,2025-04-06,conventional
4,4,Bahrain Grand Prix,2025-04-13,conventional
5,5,Saudi Arabian Grand Prix,2025-04-20,conventional
6,6,Miami Grand Prix,2025-05-04,sprint_qualifying
7,7,Emilia Romagna Grand Prix,2025-05-18,conventional
8,8,Monaco Grand Prix,2025-05-25,conventional
9,9,Spanish Grand Prix,2025-06-01,conventional
10,10,Canadian Grand Prix,2025-06-15,conventional


`EventDate` is a single day, while a weekend lasts three: the next cell checks which day of the week it corresponds to. 

In [43]:
season["EventDate"].dt.day_name().value_counts()

EventDate
Sunday      23
Saturday     1
Name: count, dtype: int64

All events but one fall on a Sunday. Let's check which event is the exception.

In [44]:
season[season["EventDate"].dt.day_name() == "Saturday"][["RoundNumber", "EventName", "EventDate"]]

,RoundNumber,EventName,EventDate
22,22,Las Vegas Grand Prix,2025-11-22


`EventDate` corresponds to the race day: 23 out of 24 events in 2025 are on a Sunday.
The only exception is the Las Vegas Grand Prix, which is held on Saturday night local time.

`EventFormat` describes the type of weekend. In 2025 two formats appear: `conventional`
and `sprint_qualifying`, the latter used for weekends that include a Sprint race.
Since F1 has changed its weekend formats over the years, the next cell counts how many
events of each format there are in every season.

In [45]:
pd.DataFrame({year: s["EventFormat"].value_counts() for year, s in schedules.items()}).fillna(0).astype(int)

,2018,2019,2020,2021,2022,2023,2024,2025,2026
EventFormat,,,,,,,,,
conventional,21,21,17,19,19,16,18,18,17
sprint,0,0,0,3,3,0,0,0,0
sprint_qualifying,0,0,0,0,0,0,6,6,6
sprint_shootout,0,0,0,0,0,6,0,0,0


Until 2020 every weekend follows the conventional format. Sprint weekends were introduced
in 2021 and have changed format over the years:

- `sprint` (2021–2022): 3 events per season
- `sprint_shootout` (2023): 6 events
- `sprint_qualifying` (2024 onwards): 6 events per season

Each format has a different set and order of sessions, so the sessions of an event cannot
be assumed in advance: they are read from the `Session1`–`Session5` columns, explored next.

### Sessions

Each event lists up to five sessions in the `Session1`–`Session5` columns. The next cell
shows them for every event of the season, to see which sessions a weekend contains
and how this depends on its format.

In [46]:
session_cols = ["Session1", "Session2", "Session3", "Session4", "Session5"]
season[["EventName", "EventFormat"] + session_cols]

,EventName,EventFormat,Session1,Session2,Session3,Session4,Session5
1,Australian Grand Prix,conventional,Practice 1,Practice 2,Practice 3,Qualifying,Race
2,Chinese Grand Prix,sprint_qualifying,Practice 1,Sprint Qualifying,Sprint,Qualifying,Race
3,Japanese Grand Prix,conventional,Practice 1,Practice 2,Practice 3,Qualifying,Race
4,Bahrain Grand Prix,conventional,Practice 1,Practice 2,Practice 3,Qualifying,Race
5,Saudi Arabian Grand Prix,conventional,Practice 1,Practice 2,Practice 3,Qualifying,Race
6,Miami Grand Prix,sprint_qualifying,Practice 1,Sprint Qualifying,Sprint,Qualifying,Race
7,Emilia Romagna Grand Prix,conventional,Practice 1,Practice 2,Practice 3,Qualifying,Race
8,Monaco Grand Prix,conventional,Practice 1,Practice 2,Practice 3,Qualifying,Race
9,Spanish Grand Prix,conventional,Practice 1,Practice 2,Practice 3,Qualifying,Race
10,Canadian Grand Prix,conventional,Practice 1,Practice 2,Practice 3,Qualifying,Race


In 2025, conventional weekends have three practice sessions, qualifying and the race.
Sprint weekends replace Practice 2 and 3 with Sprint Qualifying and the Sprint.
The race is always `Session5` and the Sprint is `Session3`, but since sprint formats
have changed over the years, the next cell checks every session combination across all seasons.

In [47]:
all_events = pd.concat(schedules.values())
all_events[["EventFormat"] + session_cols].value_counts().to_frame("Events")

,,,,,,Events
EventFormat,Session1,Session2,Session3,Session4,Session5,
conventional,Practice 1,Practice 2,Practice 3,Qualifying,Race,165
sprint_qualifying,Practice 1,Sprint Qualifying,Sprint,Qualifying,Race,18
sprint,Practice 1,Qualifying,Practice 2,Sprint,Race,6
sprint_shootout,Practice 1,Qualifying,Sprint Shootout,Sprint,Race,6
conventional,Practice 1,,,Qualifying,Race,1


One conventional event only has three sessions. Let's find out which one it is.

In [48]:
all_events[all_events["Session2"].fillna("").eq("")][["EventName", "EventDate"] + session_cols]

,EventName,EventDate,Session1,Session2,Session3,Session4,Session5
14,Emilia Romagna Grand Prix,2020-11-01,Practice 1,,,Qualifying,Race


The exception is the 2020 Emilia Romagna Grand Prix, held over a shortened two-day weekend
with a single practice session.

Across all seasons:
- the race is always `Session5`;
- the Sprint is `Session3` or `Session4` depending on the format;
- the number of sessions is not always five.

Sessions are therefore identified **by name** ("Race", "Sprint") rather than by position.

### Data availability

The last column, `F1ApiSupport`, is a boolean flag, and the name suggests whether detailed
timing data (laps, telemetry) is available for the event through the F1 API.
The next cell counts its values in every season.

In [49]:
pd.DataFrame({year: s["F1ApiSupport"].value_counts() for year, s in schedules.items()}).fillna(0).astype(int)

,2018,2019,2020,2021,2022,2023,2024,2025,2026
F1ApiSupport,,,,,,,,,
True,21,21,17,22,22,22,24,24,23


The output shows "True" for every event since 2018, including those of the current season
that have not taken place yet. It indicates that the event is covered by the F1 API,
not that its data already exists.

Which events of the current season have already taken place? To find out, `EventDate`
is compared with today's date.

In [ ]:
today = pd.Timestamp.today().normalize() #remove time component
current_season = schedules[current_year]

current_season[["RoundNumber", "EventName", "EventDate"]].assign(
    Completed=current_season["EventDate"] < today
)

,RoundNumber,EventName,EventDate,Completed
2,1,Australian Grand Prix,2026-03-08,True
3,2,Chinese Grand Prix,2026-03-15,True
4,3,Japanese Grand Prix,2026-03-29,True
5,4,Miami Grand Prix,2026-05-03,True
6,5,Canadian Grand Prix,2026-05-24,True
7,6,Monaco Grand Prix,2026-06-07,True
8,7,Barcelona Grand Prix,2026-06-14,True
9,8,Austrian Grand Prix,2026-06-28,True
10,9,British Grand Prix,2026-07-05,True
11,10,Belgian Grand Prix,2026-07-19,True


At the time of writing, 16 of the 23 events of the current season have been completed.
The remaining ones are listed in the calendar but have no data yet.

An event is therefore considered available when it is supported by the F1 API
(`F1ApiSupport`) **and** its `EventDate` is in the past. Since the comparison uses
today's date, the list of available events updates automatically as the season goes on.

## Event

The next level of the hierarchy is a single Grand Prix. As decided above, an event is
identified by year and `RoundNumber`. The 2025 Italian Grand Prix (round 16) is used
as an example.

In [52]:
event = schedules[2025].get_event_by_round(16)

print(type(event))
print(isinstance(event, pd.Series))
event

<class 'fastf1.events.Event'>
True


RoundNumber                                                   16
Country                                                    Italy
Location                                                   Monza
OfficialEventName    FORMULA 1 PIRELLI GRAN PREMIO D’ITALIA 2025
EventDate                                    2025-09-07 00:00:00
EventName                                     Italian Grand Prix
EventFormat                                         conventional
Session1                                              Practice 1
Session1Date                           2025-09-05 13:30:00+02:00
Session1DateUtc                              2025-09-05 11:30:00
Session2                                              Practice 2
Session2Date                           2025-09-05 17:00:00+02:00
Session2DateUtc                              2025-09-05 15:00:00
Session3                                              Practice 3
Session3Date                           2025-09-06 12:30:00+02:00
Session3DateUtc          

An `Event` is a pandas "Series": it contains exactly the columns of the season calendar,
i.e. it is a single row of the `EventSchedule`. Session dates are given both in local
time (with the UTC offset) and in UTC.

Beyond these fields, FastF1 adds its own methods to an `Event`. Let's list them by
filtering out everything a plain pandas Series already has.

In [53]:
[name for name in dir(event)
 if not name.startswith("_")
 and name not in dir(pd.Series)
 and name not in event.index]

['get_practice',
 'get_qualifying',
 'get_race',
 'get_session',
 'get_session_date',
 'get_session_name',
 'get_sprint',
 'get_sprint_qualifying',
 'get_sprint_shootout',
 'is_testing',
 'year']

FastF1 adds different kinds of methods to an `Event`:

- **session access**: `get_session` and shortcuts for each session type
  (`get_race`, `get_sprint`, `get_qualifying`, ...);
- **session information**: `get_session_name`, `get_session_date`;
- **other**: `is_testing()` and the `year` of the event.

The session access methods are the link to the next level of the hierarchy.
Let's see what `get_race()` returns.

In [54]:
race = event.get_race()

print(type(race))
race

<class 'fastf1.core.Session'>


2025 Season Round 16: Italian Grand Prix - Race

`get_race()` returns a "Session" object, displayed only as a label (season, round, event
and session name). To verify whether it already contains the race data, its laps are
accessed below.

In [55]:
try:
    race.laps
except Exception as error:
    print(type(error).__name__, "-", error)

DataNotLoadedError - The data you are trying to access has not been loaded yet. See `Session.load`


A `Session` is created empty: it only knows which session it refers to, and its data
must be downloaded explicitly with `Session.load()`. Creating sessions is therefore
cheap, while loading them is not.

## Session

A session contains the actual data of the race. `Session.load()` downloads it; by default
it loads everything available (laps, telemetry, weather and race control messages),
which takes some time on the first run and is then read from the cache.

In [ ]:
race.load()

core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 27)


The loading log reports that all laps of driver 27 are marked as inaccurate.
This is investigated below in the session results.

Unlike `Event`, a `Session` is not a pandas object but a container. Listing its public
attributes shows which data it holds once loaded.

In [59]:
[name for name in dir(race) if not name.startswith("_")]

['api_path',
 'car_data',
 'date',
 'drivers',
 'event',
 'f1_api_support',
 'get_circuit_info',
 'get_driver',
 'laps',
 'load',
 'name',
 'pos_data',
 'race_control_messages',
 'results',
 'session_info',
 'session_start_time',
 'session_status',
 't0_date',
 'total_laps',
 'track_status',
 'weather_data']

Some of these names are methods, others are data. For each data attribute, its type
and size are printed below.

In [60]:
for name in dir(race):
    if name.startswith("_"):
        continue
    value = getattr(race, name)
    if callable(value):
        continue
    print(f"{name:25} {type(value).__name__:20} {getattr(value, 'shape', '')}")

api_path                  str                  
car_data                  dict                 
date                      Timestamp            
drivers                   list                 
event                     Event                (23,)
f1_api_support            bool                 ()
laps                      Laps                 (974, 31)
name                      str                  
pos_data                  dict                 
race_control_messages     DataFrame            (71, 9)
results                   SessionResults       (20, 22)
session_info              dict                 
session_start_time        timedelta            
session_status            DataFrame            (5, 2)
t0_date                   Timestamp            
total_laps                int                  
track_status              DataFrame            (6, 3)
weather_data              DataFrame            (144, 8)


A loaded session holds five kinds of data:

- **metadata**: name, date, event, total number of laps and other session information;
- **participants and results**: `drivers` and `results` (one row per driver);
- **laps**: `laps`, one row per lap of each driver;
- **race context**: `track_status`, `session_status`, `race_control_messages`, `weather_data`;
- **telemetry**: `car_data` and `pos_data`, stored as dictionaries rather than tables.

Laps and telemetry are explored in the race analysis notebook. The remaining data is
explored here, starting from the results.

### Results

In [61]:
race.results

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,4781D7,red_bull,Max,Verstappen,Max Verstappen,https://media.formula1.com/d_driver_fallback_i...,,1.0,1,1.0,NaT,NaT,NaT,0 days 01:13:24.325000,Finished,25.0,53.0
4,4,L NORRIS,NOR,norris,McLaren,F47600,mclaren,Lando,Norris,Lando Norris,https://media.formula1.com/d_driver_fallback_i...,,2.0,2,2.0,NaT,NaT,NaT,0 days 00:00:19.207000,Finished,18.0,53.0
81,81,O PIASTRI,PIA,piastri,McLaren,F47600,mclaren,Oscar,Piastri,Oscar Piastri,https://media.formula1.com/d_driver_fallback_i...,,3.0,3,3.0,NaT,NaT,NaT,0 days 00:00:21.351000,Finished,15.0,53.0
16,16,C LECLERC,LEC,leclerc,Ferrari,ED1131,ferrari,Charles,Leclerc,Charles Leclerc,https://media.formula1.com/d_driver_fallback_i...,,4.0,4,4.0,NaT,NaT,NaT,0 days 00:00:25.624000,Finished,12.0,53.0
63,63,G RUSSELL,RUS,russell,Mercedes,00D7B6,mercedes,George,Russell,George Russell,https://media.formula1.com/d_driver_fallback_i...,,5.0,5,5.0,NaT,NaT,NaT,0 days 00:00:32.881000,Finished,10.0,53.0
44,44,L HAMILTON,HAM,hamilton,Ferrari,ED1131,ferrari,Lewis,Hamilton,Lewis Hamilton,https://media.formula1.com/d_driver_fallback_i...,,6.0,6,10.0,NaT,NaT,NaT,0 days 00:00:37.449000,Finished,8.0,53.0
23,23,A ALBON,ALB,albon,Williams,1868DB,williams,Alexander,Albon,Alexander Albon,https://media.formula1.com/d_driver_fallback_i...,,7.0,7,14.0,NaT,NaT,NaT,0 days 00:00:50.537000,Finished,6.0,53.0
5,5,G BORTOLETO,BOR,bortoleto,Kick Sauber,01C00E,sauber,Gabriel,Bortoleto,Gabriel Bortoleto,https://media.formula1.com/d_driver_fallback_i...,,8.0,8,7.0,NaT,NaT,NaT,0 days 00:00:58.484000,Finished,4.0,53.0
12,12,K ANTONELLI,ANT,antonelli,Mercedes,00D7B6,mercedes,Kimi,Antonelli,Kimi Antonelli,https://media.formula1.com/d_driver_fallback_i...,,9.0,9,6.0,NaT,NaT,NaT,0 days 00:00:59.762000,Finished,2.0,53.0
6,6,I HADJAR,HAD,hadjar,Racing Bulls,6C98FF,rb,Isack,Hadjar,Isack Hadjar,https://media.formula1.com/d_driver_fallback_i...,,10.0,10,19.0,NaT,NaT,NaT,0 days 00:01:03.891000,Finished,1.0,53.0


The results table has one row per driver, ordered by finishing position. It confirms
the warning raised while loading: driver 27, Nico Hülkenberg, did not start the race
and has no laps.

A few columns need care:
- `Position` is always numeric, while `ClassifiedPosition` is the official classification:
  "R" for retired (Alonso) and "W" for withdrawn (Hülkenberg).
- `Time` is the total race time for the winner and the gap to the winner for the others;
  it is not meaningful for lapped or retired drivers.
- `GridPosition` vs `Position` already tells part of the story (e.g. Hadjar from 19th to 10th).
- `Q1`–`Q3` are empty, since this is a race session.

Some columns are not needed for the analysis but will be useful later: `TeamColor`
for consistent plot colours and `HeadshotUrl` for the app.

### Race context

`track_status` records every change in the state of the track during the session
(green flag, yellow flags, safety car, ...).

In [64]:
race.track_status

,Time,Status,Message
0,0 days 00:00:00,2,Yellow
1,0 days 00:04:25.233000,1,AllClear
2,0 days 00:24:59.126000,2,Yellow
3,0 days 00:33:23.394000,1,AllClear
4,0 days 01:53:33.012000,2,Yellow
5,0 days 01:53:44.950000,1,AllClear


Each row is a change of track state: `Status` is a numeric code and `Message` its meaning.
This race only had short yellow-flag periods (`2`) between all-clear phases (`1`):
no safety car or virtual safety car was deployed. According to the FastF1 documentation,
other races may also show `4` (Safety Car), `5` (Red Flag), `6` (VSC deployed) and
`7` (VSC ending).

`Time` is not a clock time or a lap number, but the time elapsed since the start of the
session. Laps also carry their own `TrackStatus` column, explored in the race analysis notebook.

`race_control_messages` contains the official messages sent by Race Control during the
session (flags, penalties, investigations, ...). The first rows show its structure.

In [65]:
race.race_control_messages.head(10)

,Time,Category,Message,Status,Flag,Scope,Sector,RacingNumber,Lap
0,2025-09-07 12:20:01,Flag,GREEN LIGHT - PIT EXIT OPEN,None,GREEN,Track,NaN,None,1
1,2025-09-07 12:27:55,Other,PIT EXIT INCIDENT INVOLVING CAR 14 (ALO) NOTED...,None,None,None,NaN,None,1
2,2025-09-07 12:28:15,Other,PIT EXIT INCIDENT INVOLVING CAR 18 (STR) NOTED...,None,None,None,NaN,None,1
3,2025-09-07 12:30:01,Other,PIT EXIT CLOSED,None,None,None,NaN,None,1
4,2025-09-07 12:30:44,Flag,YELLOW IN TRACK SECTOR 7,None,YELLOW,Sector,7.0,None,1
5,2025-09-07 12:31:59,Other,FIA STEWARDS: PIT EXIT INCIDENT INVOLVING CAR ...,None,None,None,NaN,None,1
6,2025-09-07 12:32:06,Other,FIA STEWARDS: PIT EXIT INCIDENT INVOLVING CAR ...,None,None,None,NaN,None,1
7,2025-09-07 12:32:19,Flag,YELLOW IN TRACK SECTOR 8,None,YELLOW,Sector,8.0,None,1
8,2025-09-07 12:32:19,Flag,YELLOW IN TRACK SECTOR 9,None,YELLOW,Sector,9.0,None,1
9,2025-09-07 12:32:20,Other,DRS DISABLED IN ZONE 1,None,None,None,NaN,None,1


Unlike `track_status`, here `Time` is an actual UTC timestamp: messages start before the
race (the pit exit opens 40 minutes before the 13:00 UTC start), which is why `Lap`
is already 1. `Message` is free text, while `Flag`, `Scope`, `Sector` and `RacingNumber`
describe it in a structured way. Messages are grouped by `Category`, counted below.

In [66]:
race.race_control_messages["Category"].value_counts()

Category
Other    47
Flag     22
Drs       2
Name: count, dtype: int64

Most messages fall under `Other` (incidents, investigations, stewards' decisions, pit exit
status), followed by `Flag` and `Drs`. Since flag periods are already available in
`track_status` and in the laps data, these messages are mainly useful as annotations
(e.g. penalties or incidents to mark on a plot) rather than for the analysis itself.

`weather_data` contains the weather conditions measured at the track during the session.
Let's see its structure.

In [67]:
race.weather_data.head()

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,0 days 00:00:18.360000,26.0,44.0,997.5,False,43.5,263,2.5
1,0 days 00:01:18.329000,26.1,44.0,997.4,False,43.6,246,1.6
2,0 days 00:02:18.345000,26.1,45.0,997.3,False,43.7,210,1.5
3,0 days 00:03:18.345000,26.1,45.0,997.4,False,43.7,202,1.5
4,0 days 00:04:18.355000,26.0,43.0,997.4,False,43.7,182,1.7


Weather is sampled once per minute over the whole session, with `Time` measured from the
session start as in `track_status`. Two variables matter most for the analysis: `Rainfall`,
since a wet race changes every strategy, and `TrackTemp`, which affects tyre degradation.
A summary over the session shows whether it rained and how much the track temperature varied.

In [68]:
print("Rain during the session:", race.weather_data["Rainfall"].any())
race.weather_data[["AirTemp", "TrackTemp"]].describe()

Rain during the session: False


,AirTemp,TrackTemp
count,144.000000,144.000000
mean,26.761111,43.288194
std,0.383599,1.635211
min,25.900000,39.800000
25%,26.500000,42.375000
50%,26.800000,43.750000
75%,27.100000,44.300000
max,27.400000,45.900000


The race was dry, with stable conditions: air temperature stayed between 26 and 27 °C,
and track temperature between 40 and 46 °C. Weather is therefore unlikely to explain
differences in pace in this race, but it remains a useful context check for others.

`session_status` records the main phases of the session.

In [69]:
race.session_status

,Time,Status
0,0 days 00:00:07.141000,Inactive
1,0 days 00:57:49.893000,Started
2,0 days 02:11:14.389000,Finished
3,0 days 02:24:12.942000,Finalised
4,0 days 02:24:12.942000,Ends


The race starts at about 0:58 and finishes at 2:11 of session time: the difference
(1:13:24) matches the winner's time in the results. This also clarifies `track_status`:
the first yellow-flag periods happened before the start, and only one short yellow flag
(about 12 seconds) occurred during the race.

The session therefore includes a long pre-race phase, which explains why the context
tables start well before the first lap.

## Wrap-up

The structure of FastF1 data, from the season calendar down to a single session, is now
clear. The rules defined here for selecting seasons, events and sessions are the basis
for the race analysis and for the app.